<a href="https://colab.research.google.com/github/Lau-Tisca/FlyRank_ML_1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lau-Tisca/FlyRank_ML_1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

This notebook constructs the core feature vector on a mid-panel month (`month=2026-03`), documents each feature's meaning and availability cutoff, performs an adversarial leakage hunt (temporal boundaries and deliberate leak attacks), and details excluded fields.

## 1. Build the feature vector

### Methodology & Feature Construction
* **Observation Grain**: One row represents one pseudonymized content item (`content_hash_id`) evaluated over a 15-day pre-decision window.
* **Feature Window ($T_{\text{feat}}$)**: March 1, 2026 to March 15, 2026 (`report_date BETWEEN '2026-03-01' AND '2026-03-15'`).
* **Outcome Window ($T_{\text{target}}$)**: March 16, 2026 to March 31, 2026 (`report_date BETWEEN '2026-03-16' AND '2026-03-31'`).
* **Target Label Definition ($\text{Is\_Opportunity}$)**: Binary indicator set to 1 if post-decision clicks drop by $>20\%$ relative to pre-decision volume ($T_{\text{target}} < 0.80 \times T_{\text{feat}}$).
* **Missing Value & Categorical Strategy**:
  - Counts and engagement (`gsc_impressions`, `gsc_clicks`, `ga4_sessions`): Missing values filled with `0`.
  - Position (`gsc_avg_position`): Unranked or missing SERP values filled with `50.0` (unranked baseline floor).
  - Skewed metrics: Transformed via $\log(1 + x)$ to normalize heavy-tailed search traffic distributions.

In [4]:
import os
import duckdb
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# 1. Connect DuckDB & Authenticate Hugging Face
con = duckdb.connect()
hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass

if hf_token:
    con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# 2. Extract Features (Days 1-15) and Target (Days 16-31)
query = f"""
WITH feat AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS raw_impressions,
        SUM(gsc_clicks) AS raw_clicks,
        AVG(gsc_avg_position) AS raw_avg_position,
        SUM(ga4_sessions) AS raw_ga4_sessions,
        COUNT(DISTINCT report_date) AS feat_active_days
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
      AND ga4_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) >= 50
),
target AS (
    SELECT
        content_hash_id,
        SUM(gsc_clicks) AS target_clicks
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'
    GROUP BY content_hash_id
)
SELECT
    f.content_hash_id,
    f.client_hash_id,
    f.raw_impressions,
    f.raw_clicks,
    f.raw_avg_position,
    f.raw_ga4_sessions,
    f.feat_active_days,
    COALESCE(t.target_clicks, 0) AS target_clicks_future,
    CASE WHEN COALESCE(t.target_clicks, 0) < (0.80 * f.raw_clicks) THEN 1 ELSE 0 END AS is_opportunity
FROM feat f
LEFT JOIN target t ON f.content_hash_id = t.content_hash_id
"""

df_features = con.sql(query).df()

# 3. Apply Feature Transformations and Imputations
df_features['log_impressions'] = np.log1p(df_features['raw_impressions'].fillna(0))
df_features['log_clicks'] = np.log1p(df_features['raw_clicks'].fillna(0))
df_features['avg_position'] = df_features['raw_avg_position'].fillna(50.0)
df_features['ctr_feat'] = ((df_features['raw_clicks'] / (df_features['raw_impressions'] + 1e-5)) * 100.0).fillna(0.0)
df_features['log_ga4_sessions'] = np.log1p(df_features['raw_ga4_sessions'].fillna(0))
df_features['feat_active_days'] = df_features['feat_active_days'].fillna(0)

feature_cols = ['log_impressions', 'log_clicks', 'avg_position', 'ctr_feat', 'log_ga4_sessions', 'feat_active_days']

print(f"✓ Feature Vector Built: {len(df_features):,} rows across {df_features['client_hash_id'].nunique()} unique clients.")
print(f"✓ Feature Columns: {feature_cols}")
display(df_features[['content_hash_id'] + feature_cols + ['is_opportunity']].head(5))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Feature Vector Built: 22,123 rows across 26 unique clients.
✓ Feature Columns: ['log_impressions', 'log_clicks', 'avg_position', 'ctr_feat', 'log_ga4_sessions', 'feat_active_days']


,content_hash_id,log_impressions,log_clicks,avg_position,ctr_feat,log_ga4_sessions,feat_active_days,is_opportunity
0,content_bd1905f7fdd727d0,5.117994,1.609438,10.295629,2.409638,1.945910,6,1
1,content_49727633c22525ba,5.231109,1.945910,8.662374,3.225806,2.772589,10,0
2,content_4f6c994af9bc9458,4.672829,0.000000,9.487270,0.000000,1.386294,3,0
3,content_5147a94966f9a69b,5.616771,0.000000,8.570975,0.000000,2.397895,6,0
4,content_1d9af290412f7ae3,4.110874,1.098612,8.174444,3.333333,1.386294,3,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature Name | Meaning / Business Intent | Missing Value Handling | Available-When? (Decision Moment) |
| :--- | :--- | :--- | :--- |
| `log_impressions` | Log-transformed aggregate GSC impressions ($\log(1 + \text{Impressions})$), representing total search exposure. | Filled with `0.0`. | **Pre-decision:** Fully knowable at Day 15 cutoff from historical daily GSC logs. |
| `log_clicks` | Log-transformed aggregate GSC clicks ($\log(1 + \text{Clicks})$), representing pre-intervention baseline traffic. | Filled with `0.0`. | **Pre-decision:** Measured strictly across Days 1–15 before any editorial intervention. |
| `avg_position` | Mean SERP ranking across the 15-day window. Lower numbers represent higher rank on search result pages. | Filled with `50.0` (unranked baseline penalty). | **Pre-decision:** Computed solely from queries active during Days 1–15. |
| `ctr_feat` | Click-through rate ($\text{Clicks} / \text{Impressions} \times 100$). Measures snippet efficiency. | Filled with `0.0%`. | **Pre-decision:** Both numerator and denominator close at end of Day 15. |
| `log_ga4_sessions` | Log-transformed GA4 verified on-site user sessions. Measures on-site engagement. | Filled with `0.0`. | **Pre-decision:** Tracked up to Day 15 for mature properties. |
| `feat_active_days` | Number of distinct active tracking dates in the feature window ($1 \dots 15$). | Filled with `0`. | **Pre-decision:** Complete historical log state known at Day 15. |

In [5]:
# Feature Profile & Availability Summary
profile_rows = []
for col in feature_cols:
    profile_rows.append({
        'Feature': col,
        'Dtype': str(df_features[col].dtype),
        'Null Count': int(df_features[col].isnull().sum()),
        'Min': round(float(df_features[col].min()), 3),
        'Mean': round(float(df_features[col].mean()), 3),
        'Max': round(float(df_features[col].max()), 3),
        'Available Cutoff': '2026-03-15 23:59:59'
    })

df_profile = pd.DataFrame(profile_rows)
print("--- Feature Vector Summary & Availability Audit ---")
display(df_profile)

--- Feature Vector Summary & Availability Audit ---


,Feature,Dtype,Null Count,Min,Mean,Max,Available Cutoff
0,log_impressions,float64,0,3.932,6.046,11.993,2026-03-15 23:59:59
1,log_clicks,float64,0,0.000,1.235,7.782,2026-03-15 23:59:59
2,avg_position,float64,0,0.000,15.080,85.899,2026-03-15 23:59:59
3,ctr_feat,float64,0,0.000,0.796,17.722,2026-03-15 23:59:59
4,log_ga4_sessions,float64,0,0.000,2.437,7.093,2026-03-15 23:59:59
5,feat_active_days,int64,0,1.000,5.300,15.000,2026-03-15 23:59:59


## 3. The leakage hunt

### Adversarial Testing Strategy
To prove zero data leakage into the decision space, we execute three validation attacks:
1. **Temporal Isolation Verification**: Verifying that the maximum report date in the feature extraction query strictly equals `2026-03-15`.
2. **Correlation Scan**: Checking feature-to-target Pearson correlations to ensure no single feature acts as a near-perfect label proxy ($|r| > 0.85$).
3. **The Deliberate Leakage Trap**: Injecting a known future-window variable (`leaked_future_ratio = target_clicks / pre_clicks`), observing the artificial metric spike, and confirming its removal from production matrices.

In [6]:
# 1. Temporal Boundary Check
date_check_query = f"""
SELECT MAX(report_date) AS max_feat_date
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15';
"""
max_date = con.sql(date_check_query).df().iloc[0, 0]
print(f"1. Temporal Check: Max feature extraction date = {max_date}")
assert str(max_date)[:10] == '2026-03-15', f"Temporal Leakage: Date {max_date} exceeds cutoff!"
print("   ✓ Passed: Feature extraction strictly terminates at Day 15.")

# 2. Linear Correlation Scan with Target
corrs = df_features[feature_cols].apply(lambda c: c.corr(df_features['is_opportunity']))
print("\n2. Feature-to-Target Pearson Correlation Audit:")
for feat, r in corrs.items():
    print(f"   • {feat:<18}: r = {r:+.4f}")
    assert abs(r) < 0.85, f"Suspicious target leak in {feat}: r = {r}"
print("   ✓ Passed: All features show honest, non-deterministic correlations.")

# 3. Deliberate Leakage Trap Experiment
X_honest = df_features[feature_cols].values
y = df_features['is_opportunity'].values

model_honest = LogisticRegression(max_iter=500, random_state=42).fit(X_honest, y)
honest_auc = roc_auc_score(y, model_honest.predict_proba(X_honest)[:, 1])

# Inject Future Outcome Leak
df_features['leaked_future_ratio'] = df_features['target_clicks_future'] / (df_features['raw_clicks'] + 1.0)
X_leaked = df_features[feature_cols + ['leaked_future_ratio']].values

model_leaked = LogisticRegression(max_iter=500, random_state=42).fit(X_leaked, y)
leaked_auc = roc_auc_score(y, model_leaked.predict_proba(X_leaked)[:, 1])

# Purge the leak
df_features.drop(columns=['leaked_future_ratio'], inplace=True)

print(f"\n3. Deliberate Leakage Trap Results:")
print(f"   • Honest Model ROC AUC:        {honest_auc:.4f}")
print(f"   • Leaked Target Model ROC AUC: {leaked_auc:.4f} (Spike: +{leaked_auc - honest_auc:.4f})")
print("   ✓ Verified: Leaked column isolated, tested, and dropped from production frame.")

1. Temporal Check: Max feature extraction date = 2026-03-15 00:00:00
   ✓ Passed: Feature extraction strictly terminates at Day 15.

2. Feature-to-Target Pearson Correlation Audit:
   • log_impressions   : r = +0.0239
   • log_clicks        : r = +0.2034
   • avg_position      : r = -0.0245
   • ctr_feat          : r = +0.2524
   • log_ga4_sessions  : r = -0.0036
   • feat_active_days  : r = +0.0856
   ✓ Passed: All features show honest, non-deterministic correlations.

3. Deliberate Leakage Trap Results:
   • Honest Model ROC AUC:        0.7087
   • Leaked Target Model ROC AUC: 0.9827 (Spike: +0.2740)
   ✓ Verified: Leaked column isolated, tested, and dropped from production frame.


## 4. What I excluded and why

### Excluded Fields & Rationales
1. `client_hash_id` & `content_hash_id` (as numeric model features): **Excluded** to prevent entity memorization and avoid learning domain-specific noise that fails on unseen client domains.
2. `trend_pct` & `trend_direction` (from starter sample tables): **Excluded** because these fields were aggregated over the entire monthly partition, introducing direct future outcome leakage.
3. `target_clicks_future` & post-Day-15 metrics: **Excluded** because outcome window statistics are physically unknowable at the Day 15 decision moment.
4. Non-GA4 instrumented rows (`ga4_data_available IS NOT TRUE`): **Excluded** to prevent misclassifying missing analytics telemetry as zero-traffic decay.
5. Low-volume search tails (`raw_impressions < 50`): **Excluded** to eliminate high-variance statistical noise from unranked queries.

In [7]:
# Exclusion Verification Check
banned_fields = ['trend_pct', 'trend_direction', 'target_clicks_future', 'leaked_future_ratio', 'client_hash_id']
active_feature_columns = feature_cols

violations = [col for col in active_feature_columns if col in banned_fields]
print(f"Checking for banned/leaked fields in active feature set...")
print(f"Active Features: {active_feature_columns}")
print(f"Violations Found: {len(violations)}")

assert len(violations) == 0, f"Integrity Failure: Leaked fields present: {violations}"
print("✓ Exclusion Integrity Confirmed: Zero banned or lookahead fields present.")

Checking for banned/leaked fields in active feature set...
Active Features: ['log_impressions', 'log_clicks', 'avg_position', 'ctr_feat', 'log_ga4_sessions', 'feat_active_days']
Violations Found: 0
✓ Exclusion Integrity Confirmed: Zero banned or lookahead fields present.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.